# FeCl3 diffusing into cement paste (xGEM2MT, 'A' mode without flow)

An FeCl3 solution is in contact with a hydrated Portland cement paste. There is no flow: the chloride and the iron
move into the paste by **diffusion** only, and react with the cement phases (portlandite, C-S-H, AFt/AFm phases).

The model is a 1-D column, as in the calcite-dolomite examples, with velocity 0. The geometry, the porosity and the
diffusion coefficient come from the CEBAMA low-pH concrete / clay benchmark (Idiart et al., 2020, *Appl. Geochem.* 115,
104562, Table 1). The chemistry is from the **PCCEM26** project (TDB 2020 + Cemdata26), with chlorine added: the project
`PCCEM26_Cl` has the same system as PCCEM26 and the element Cl with its aqueous species and the Cl-bearing cement phases
(Friedel's salt, Kuzel's salt). GEMS3K files of the system: folder `PCCEM26Cl-CASH/` (one dbr file).

**Two C-(A)-S-H models.** The notebook uses the **CASH+AlFe** solid solution (Ca-Al-Si-Fe), which takes up Al and Fe. The folder
`PCCEM26Cl/` has the same system with the **CSHQ** model, which does not take up Al and Fe. To use it, change the `system` line
in the first code cell. The bulk composition and the transport properties are the same, so the results can be compared. The text
in the notes (section 5) and the saved results are for CASH+AlFe; with CSHQ, Friedel's salt can form and unreacted clinker stays in the solid.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from xgems import ChemicalEngine, Material
from xgem2mt import MassTransport, TransportProperties

# One GEMS3K system: hydrated OPC paste (100 g OPC + 40 g water), 20 C, 1 bar. Choose the C-(A)-S-H model:
system = "PCCEM26Cl-CASH/OPC-dat.lst"     # CASH+AlFe: takes up Al and Fe
#system = "PCCEM26Cl/OPC-dat.lst"         # CSHQ: does not take up Al and Fe (uncomment this line and comment the one above)
gems = ChemicalEngine(system)        # gives Material the element names and molar masses
YEAR = 365.25 * 86400                # s

## 1. Compositions: hydrate the paste once, then split it

**Paste.** The column is a cement paste: OPC clinker hydrated with pure water (water/cement = 0.4, mass). The oxide masses (g)
are the clinker of the PCCEM26 compos `OPC:MIN:CI-2022_clinker_` (about 99 g). We add each oxide to a `Material`, add the
water, and calculate the equilibrium once with GEM. (A mortar with 70 g of inert sand per 100 g of clinker in the column, at porosity 0.04,
was about 3 times slower and had GEM failures, so the column has no sand; see the note on porosity in section 5.)

**Solids and pore solution.** From this result we take the element amounts of the aqueous phase (the pore solution) and of
all other phases (the solids: the hydrates and the rest of the clinker). Most of the water is bound in the hydrates; only a few g remain as pore solution. The nodes
are then set up from these two parts with `setSystem(pore_solution=..., solids=...)`: each node gets the pore solution as
it is, and the program adds as much solid as the transport porosity of the node needs. We take the porosity of the paste
itself (from the GEM result, printed below), so each node is the hydrated paste (section 2). The porosity is the volume of
the pore solution divided by the volume of pore solution and solids; a gas phase is left out, because GEMS gives it the volume of a
free gas.

**Inert sand.** The chemical system has an element `Inert` with one solid phase `Inert` (the molar mass and volume of quartz,
but it does not react), as the inert solid of the calcite-dolomite example. 

**FeCl3 solution in inert sand.** The inflow nodes hold the FeCl3 solution (0.1 mol per kg of water) in the same sand. The
sand fills the solid volume of the inflow nodes, so these nodes have the same porosity as the paste.

> **Note.** In 'A' mode only the aqueous phase moves, and the node amounts are conserved only if **all nodes have the same
> mass of water**. `setSystem` does this for you: the first call with `pore_solution` (here system 0, the FeCl3 solution with
> 1 kg of water) sets the water mass of the nodes, and the next calls scale their solution to it.

In [ ]:
T, P = gems.temperature(), gems.pressure()   # K, Pa of the system file
aq = gems.aqueousPhaseName()

# OPC clinker (g), PCCEM26 compos OPC:MIN:CI-2022_clinker_, 99.05 g in total
clinker = Material(gems, "OPC clinker")
clinker.add("CaO", 62.715, "g")
clinker.add("SiO2", 19.305, "g")
clinker.add("Al2O3", 4.112, "g")
clinker.add("Fe2O3", 5.777, "g")
clinker.add("MgO", 0.558, "g")
clinker.add("K2O", 0.768, "g")
clinker.add("Na2O", 0.271, "g")
clinker.add("SO3", 3.561, "g")
clinker.add("CO2", 1.982, "g")

water = Material(gems, "pure water")
water.add("H2O", 0.4 * clinker.mass_kg(), "kg")   # w/c = 0.4

# Hydrate the paste: one GEM calculation
paste = clinker + water
ret = gems.equilibrate(T, P, paste)
assert gems.converged(), f"paste hydration failed (code {ret})"

# Split the result into pore solution and solids (element amounts, mol)
b_pore = gems.elementAmountsInPhase(aq)        # pore solution: water + dissolved ions
b_solid = gems.elementAmounts() - b_pore       # all the rest: the hydrates

# Porosity of the paste: pore solution volume / (pore solution + solids). GEMS gives a gas phase the volume of a free gas
# at the pressure of the system, so a trace of gas can add much volume: leave the gas phase out (here it is absent).
gas = gems.gasPhaseName()
v_gas = gems.phaseVolume(gas) if gas else 0.0
porosity = gems.phaseVolume(aq) / (gems.systemVolume() - v_gas)
print(f"clinker {clinker.mass_kg()*1e3:.1f} g + water {water.mass_kg()*1e3:.1f} g "
      f"-> pore solution {gems.phaseMass(aq)*1e3:.2f} g, pH {gems.pH():.2f}, porosity of the paste {porosity:.3f}")

fecl3 = Material(gems, "FeCl3 solution")       # any amount: it is scaled to the water of the paste nodes
fecl3.add("H2O", 1.0, "kg")
fecl3.add("FeCl3", 0.1, "mol")                   # 0.1 mol/kg, pH about 1.9

sand = Material(gems, "inert sand")              # any amount: it fills the solid volume of the inflow nodes
sand.add({"Inert": 1.0}, 1.0, "kg")


## 2. Transport properties

| | CEBAMA benchmark (Table 1, low-pH concrete) | here |
|---|---|---|
| Porosity ε | 0.04 | 0.167, the porosity of the paste (section 1) |
| Pore diffusion coefficient Dp (m²/s) | 2.5e-11 | 2.5e-11 |
| Effective diffusion coefficient De = Dp·ε (m²/s) | 1.0e-12 | 4.2e-12 |
| Element size (m) | 0.02 | 0.002 (the front is steeper than in the benchmark) |

**Which D does 'A' mode use?** The program computes `D = dispersivity·velocity + diffusivity·porosity/tortuosity` and moves
amounts between the nodes with `dn = D·dt/dx²`. All nodes hold the same mass of pore water, so this is a diffusion equation for
the **pore water concentration**: `D` must be the pore diffusion coefficient `Dp` (2.5e-11 m²/s), not the effective `De`.
The porosity only reduces the free-water diffusion coefficient in this formula. We take `diffusivity = 1e-9` m²/s (free
water) and calculate the tortuosity so that `D = Dp`: `tortuosity = 1e-9 · ε / Dp` (6.7 for ε = 0.167; the benchmark
values give 1.6 for ε = 0.04). So D, the time step and the diffusion of the pore water are as in the benchmark.

**Why the porosity of the paste, not 0.04?** `setSystem` adds as much solid to each node as the porosity needs. With 0.04 a node has
1 kg of pore water and about 59 kg of hydrates (1:57 by mass; the hydrated paste has 1:12). For GEM this is a hard system: many
calculations then needed 20 000 to 120 000 iterations (up to about 3 minutes for one node) and some failed. With the porosity of the paste,
each node is the hydrated paste itself.

Velocity is 0 and the dispersivity is 0 (pure diffusion). The time step is then limited by diffusion only,
`dx²/(4 D) = 4e4 s`.

In [ ]:
N, L = 30, 0.06                       # nodes, column length (m): node length 0.002 m
task = MassTransport("A", N)
task.setName("FeCl3 diffusion into cement paste")
task.setTau(0.0, 0.25 * YEAR, 3.9e4)  # start, stop, step (s): the step is a little smaller than the program time step
task.setSpatialDimensions(L)
task.setFluidAdvectionVelocity(0.0)   # no flow
Dp = 2.5e-11                          # m2/s, pore diffusion coefficient (CEBAMA benchmark, section 2)
diffusivity = 1e-9                    # m2/s, free water
tortuosity = diffusivity * porosity / Dp   # so that 'A' mode uses D = diffusivity * porosity / tortuosity = Dp
task.setInitialNodeEffectivePorosity(porosity)   # the porosity of the paste (section 1)
task.setInitialDispersivity(0.0)
task.setInitialDiffusivity(diffusivity)
task.setInitialTortuosityFactor(tortuosity)
print(f"porosity {porosity:.3f}, tortuosity {tortuosity:.2f}, D = {diffusivity * porosity / tortuosity:.2e} m2/s")
task.initNodes(system)                # one dbr file: all nodes start from it

# System 0 first: its water (1 kg in fecl3) sets the water mass of all nodes; system 1 is scaled to it
task.setSystem(0, pore_solution=fecl3, solids=sand)      # system 0: inflow, nodes 0 and 1 (fixed), in sand
task.setSystem(1, pore_solution=b_pore, solids=b_solid)  # system 1: the paste, all other nodes
print(task)

`print(task)` shows what the program will do: node 0-1 are the fixed inflow, nodes 2 to 28 the paste, and the last node is a
fixed outlet with the paste composition. The step list also shows the time step that the program uses.

## 3. Run and record profiles

The monitoring function runs after each time step. We store the profiles at four times.

In [ ]:
names = [gems.phaseName(i) for i in range(gems.numPhases())]
elements = [gems.elementName(i) for i in range(gems.numElements())]
iCl, iFe = elements.index("Cl"), elements.index("Fe")
show = [0.01, 0.05, 0.1, 0.25]        # years
snap = {}

def monitor(engine, step, t):
    for y in list(show):
        if t >= y * YEAR:
            show.remove(y)
            aq = np.array([engine.phaseMass(n, "aq_sit") for n in range(N)])
            snap[y] = dict(
                t=t / YEAR,
                pH=np.array([engine.pH(n) for n in range(N)]),
                Cl=np.array([engine.elementAmountsInPhase(n, "aq_sit")[iCl] for n in range(N)]) / aq,
                Fe=np.array([engine.elementAmountsInPhase(n, "aq_sit")[iFe] for n in range(N)]) / aq,
                ph={p: np.array([engine.phaseAmount(n, p) for n in range(N)]) for p in names})

task.setMonitoring(monitor)
t0 = time.time()
task.runMassTransport()
print(f"run time {time.time() - t0:.0f} s, {len(snap)} profiles")
print(task.gemFailures())

## 4. Results

In [ ]:
dx = L / N
x = (np.arange(N) + 0.5) * dx * 100   # node centres, cm
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for y, s in snap.items():
    ax[0].plot(x[2:], s["Cl"][2:], label=f"{s['t']:.2f} a")
    ax[1].plot(x[2:], s["pH"][2:])
    ax[2].plot(x[2:], s["Fe"][2:])
ax[0].set(xlabel="distance (cm)", ylabel="dissolved Cl (mol/kg water)", title="chloride"); ax[0].legend()
ax[1].set(xlabel="distance (cm)", ylabel="pH", title="pH")
ax[2].set(xlabel="distance (cm)", ylabel="dissolved Fe (mol/kg water)", title="iron", yscale="log")
plt.tight_layout(); plt.show()

In [ ]:
last = snap[max(snap)]
# change of each solid phase from the unreacted paste (the last paste node), in mol per node
change = {p: last["ph"][p] - last["ph"][p][-2] for p in names if p != "aq_sit" and abs(last["ph"][p] - last["ph"][p][-2]).max() > 2e-4}
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for p, d in change.items():
    ax[0].plot(x[2:], d[2:], label=p)
ax[0].set(xlabel="distance (cm)", ylabel="change from the paste (mol per node)", title=f"phase changes after {last['t']:.2f} a")
ax[0].legend(fontsize=7)
for p in ("C4ACl2H10", "Ca(OH)2(cr)"):
    ax[1].plot(x[2:], last["ph"][p][2:], label=p)
ax[1].set(xlabel="distance (cm)", ylabel="mol per node", title="Friedel's salt (C4ACl2H10) and portlandite")
ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. Notes

- **What happens.** The paste buffers the acid FeCl3 solution: only the first 1 to 2 cm change (portlandite dissolves; with CASH+ no Friedel's
  salt forms), the chloride goes further as dissolved Cl. The iron is taken up near the inlet; dissolved Fe is about 1e-8 mol/kg at pH 13.
- **Limits of this setup.** 'A' mode moves only the aqueous phase and needs the same aqueous mass in all nodes (known limit).
  The solids stay in place. The porosity (that of the paste) is a transport value; it is not changed by the precipitation and dissolution
  in the nodes (the benchmark has porosity-diffusion coupling only in its sensitivity case S2).
- **Low porosity or much solid per water.** With a porosity much below 0.1, or another setup with much solid per kg of pore water,
  GEM can become slow and unreliable for cement systems: at porosity 0.04 (1 kg water, 59 kg solids per node) single node
  calculations took up to about 3 minutes, some failed or gave a wrong result (pH 14.2 instead of 13.7), and a run of 0.25 years would take
  about 17 hours. The same nodes with 1 kg more water solve in about 0.1 s. If you need a low porosity, keep the solid/water
  ratio of a real material, check `task.gemFailures()` after the run, and test a few nodes first.
- **All solutes have the same diffusion coefficient**, as in the benchmark reference case (no electrochemical coupling).
- **Run time.** 0.25 years (198 steps, 30 nodes) take about 6.5 minutes. A longer time or finer nodes need more steps:
  the time step goes with `dx²`. A few GEM calculations can fail (2 of about 5800 in a test run, `task.gemFailures()`): such a
  node keeps its state from before the step, and the run goes on.
- **Chemistry.** The system file is one equilibrium export. To change the phases or the species (for example more Fe
  phases), open the project `PCCEM26_Cl` in GEM-Selektor and export a new system.